In [1]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
folder_path = '/Users/udaykaransingh/Desktop/study/Projects/Project_Non-linear_NN/UTKFace'

In [5]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [7]:
len(age)

23708

In [9]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})
df.shape

(23708, 3)

In [11]:
df.head()

,age,gender,img
0,9,1,9_1_2_20161219204347420.jpg.chip.jpg
1,36,0,36_0_1_20170117163203851.jpg.chip.jpg
2,86,1,86_1_0_20170120225751953.jpg.chip.jpg
3,26,1,26_1_0_20170116171048641.jpg.chip.jpg
4,1,1,1_1_2_20161219154612988.jpg.chip.jpg


In [13]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [15]:
train_df.shape

(20000, 3)

In [17]:
test_df.shape

(3708, 3)

In [19]:
import tensorflow as tf

def preprocess(path, age, gender):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (200, 200))
    img = img / 255.0
    return img, (age, gender)


In [21]:
train_ds = (
    tf.data.Dataset.from_tensor_slices((
        train_df['img'].apply(lambda x: folder_path + '/' + x).values,
        train_df['age'].values,
        train_df['gender'].values
    ))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

In [23]:
test_ds = (
    tf.data.Dataset.from_tensor_slices((
        test_df['img'].apply(lambda x: folder_path + '/' + x).values,
        test_df['age'].values,
        test_df['gender'].values
    ))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

# Using Transfer Learning

In [27]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [29]:
resnet = ResNet50(include_top=False, 
                  input_shape=(200,200,3), 
                  weights='imagenet')

In [31]:
resnet = ResNet50(include_top=False, 
                  input_shape=(200,200,3),
                  weights='imagenet')

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [33]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [35]:
model.compile(
    optimizer='adam',
    loss=['mae', 'binary_crossentropy'],   # ✅ list matches y[:,0], y[:,1]
    metrics=[['mae'], ['accuracy']],       # ✅ per-output metrics
    loss_weights=[1, 99]
)


In [ ]:
model.fit(
    train_ds,
    epochs=10,
    validation_data=test_ds
)


Epoch 1/10
 62/313 ━━━━━━━━━━━━━━━━━━━━ 5:31 1s/step - age_loss: 19.2280 - age_mae: 19.2280 - gender_accuracy: 0.4703 - gender_loss: 5.0608 - loss: 520.2449